In [1]:
import os
import re
import pickle
import numpy as np
import pandas as pd

def _is_xyz_array(a):
    if not isinstance(a, np.ndarray):
        return False
    if a.ndim != 2:
        return False
    return (a.shape[1] == 3) and np.issubdtype(a.dtype, np.number)

def _flatten_dict(d, prefix=""):
    """Flatten nested dict-like objects into path->value mapping."""
    out = {}
    if isinstance(d, dict):
        for k, v in d.items():
            key = f"{prefix}.{k}" if prefix else str(k)
            out.update(_flatten_dict(v, key))
    else:
        out[prefix] = d
    return out

def _pick_candidate(flat, want="xyz"):
    """
    Try to locate likely fields in flattened pickle object:
      - xyz: Nx3 numeric
      - regions: N strings or list-like
      - depths: N numeric
    """
    keys = list(flat.keys())

    if want == "xyz":
        # Prefer keys containing xyz/coord/points and holding Nx3 arrays
        preferred = [k for k in keys if re.search(r"(xyz|coord|point|trajectory|track)", k, re.I)]
        for k in preferred + keys:
            v = flat[k]
            if isinstance(v, np.ndarray) and _is_xyz_array(v):
                return k, v
        return None, None

    if want == "regions":
        # Prefer acronym/region/label keys
        preferred = [k for k in keys if re.search(r"(acronym|region|label|area)", k, re.I)]
        for k in preferred + keys:
            v = flat[k]
            # accept list/array of strings length N
            if isinstance(v, (list, tuple, np.ndarray)):
                if len(v) > 0 and all(isinstance(x, (str, np.str_)) for x in v):
                    return k, np.array(v, dtype=str)
        return None, None

    if want == "depths":
        preferred = [k for k in keys if re.search(r"(depth|position|dist|along)", k, re.I)]
        for k in preferred + keys:
            v = flat[k]
            if isinstance(v, np.ndarray) and v.ndim == 1 and np.issubdtype(v.dtype, np.number):
                return k, v.astype(float)
            if isinstance(v, (list, tuple)) and len(v) > 0 and all(isinstance(x, (int, float, np.number)) for x in v):
                return k, np.array(v, dtype=float)
        return None, None

    return None, None

def _ensure_100_points(xyz, regions, n_points=100):
    """
    Resample along cumulative distance to a fixed number of points.
    Regions are resampled using nearest neighbor on the same parameterization.
    """
    xyz = np.asarray(xyz, dtype=float)
    if xyz.shape[0] < 2:
        raise ValueError("Track has <2 points; cannot resample.")
    # cumulative distance parameter
    diffs = np.diff(xyz, axis=0)
    seglen = np.sqrt((diffs**2).sum(axis=1))
    s = np.concatenate([[0.0], np.cumsum(seglen)])
    if s[-1] == 0:
        # all points identical
        s = np.linspace(0, 1, xyz.shape[0])

    s_new = np.linspace(s[0], s[-1], n_points)

    # interpolate xyz
    xyz_new = np.vstack([
        np.interp(s_new, s, xyz[:, 0]),
        np.interp(s_new, s, xyz[:, 1]),
        np.interp(s_new, s, xyz[:, 2]),
    ]).T

    # resample regions by nearest original index
    if regions is None:
        regions_new = np.array(["root"] * n_points, dtype=str)
    else:
        regions = np.asarray(regions, dtype=str)
        # map each s_new to nearest s index
        idx = np.searchsorted(s, s_new, side="left")
        idx = np.clip(idx, 0, len(s) - 1)
        # fix to nearest (left/right)
        idx_left = np.clip(idx - 1, 0, len(s) - 1)
        choose_left = (np.abs(s_new - s[idx_left]) <= np.abs(s_new - s[idx]))
        idx = np.where(choose_left, idx_left, idx)
        regions_new = regions[idx]

    return xyz_new, regions_new

def convert_probe_pkl(
    probe_pkl_path,
    out_dir,
    n_points=100,
    assume_um=True,
    force_xyz_key=None,
    force_regions_key=None,
    force_depths_key=None,
):
    """
    Converts HERBS probe.pkl into MATLAB-friendly per-shank CSV + NPY.
    Output:
      - histology_shank{sh}.csv: columns Position, RegionAcronym
      - trackcoordinates_shank{sh}.npy: Nx3
    """

    os.makedirs(out_dir, exist_ok=True)

    with open(probe_pkl_path, "rb") as f:
        obj = pickle.load(f)

    # HERBS may store probe data as dict or object; try dict view first
    if not isinstance(obj, dict):
        # best effort: pull __dict__
        if hasattr(obj, "__dict__"):
            obj = obj.__dict__
        else:
            raise TypeError(f"probe.pkl loaded type {type(obj)} not dict-like. Inspect manually.")

    flat = _flatten_dict(obj)

    print("\n=== probe.pkl flattened keys (first 60) ===")
    for k in list(flat.keys())[:60]:
        v = flat[k]
        shape = getattr(v, "shape", None)
        print(f"{k} | type={type(v).__name__} | shape={shape}")

    # Many tools store multi-shank as list of shank dicts, or dict with shank keys.
    # Heuristics: find shank-like containers
    shank_containers = []
    for k, v in obj.items():
        if re.search(r"(shank|probe|tracks?)", str(k), re.I) and isinstance(v, (list, tuple)):
            # list of shanks
            if len(v) > 0 and isinstance(v[0], (dict,)):
                shank_containers = v
                break

    # If not found, treat the whole object as single shank
    if not shank_containers:
        shank_containers = [obj]

    outputs = []
    for sh_i, sh in enumerate(shank_containers, start=1):
        sh_flat = _flatten_dict(sh if isinstance(sh, dict) else getattr(sh, "__dict__", {}))

        # Find xyz
        if force_xyz_key is not None:
            xyz = sh_flat.get(force_xyz_key, None)
            xyz_key = force_xyz_key
        else:
            xyz_key, xyz = _pick_candidate(sh_flat, "xyz")

        # Find regions
        if force_regions_key is not None:
            regions = sh_flat.get(force_regions_key, None)
            regions_key = force_regions_key
        else:
            regions_key, regions = _pick_candidate(sh_flat, "regions")

        # Find depths/positions (optional)
        if force_depths_key is not None:
            depths = sh_flat.get(force_depths_key, None)
            depths_key = force_depths_key
        else:
            depths_key, depths = _pick_candidate(sh_flat, "depths")

        if xyz is None:
            raise ValueError(
                f"Could not find Nx3 coordinates for shank {sh_i}. "
                f"Re-run and set force_xyz_key to one of these keys:\n"
                + "\n".join(list(sh_flat.keys())[:200])
            )

        xyz = np.asarray(xyz, dtype=float)

        # If regions missing, still export; alignatlasdata will map 'root' segments.
        if regions is not None:
            regions = np.asarray(regions, dtype=str)
            # If regions is length mismatch, ignore and fill later.
            if len(regions) != xyz.shape[0]:
                print(f"[WARN] Shank {sh_i}: regions length {len(regions)} != xyz rows {xyz.shape[0]}. Will resample/fill.")
                regions = None

        # If depths exist and match xyz, we can use them; else compute cumulative distance.
        if depths is not None and len(depths) == xyz.shape[0]:
            pos = np.asarray(depths, dtype=float)
        else:
            diffs = np.diff(xyz, axis=0)
            seglen = np.sqrt((diffs**2).sum(axis=1))
            pos = np.concatenate([[0.0], np.cumsum(seglen)])
            # Convert to microns if likely in mm
            # This is heuristic; your atlas coords are often in microns already.
            if assume_um:
                # If total length < 20, it might be in mm (e.g., 3.5 mm)
                if pos[-1] < 50:
                    pos = pos * 1000.0  # mm->um
            else:
                pass

        # Resample to fixed points (alignatlasdata expects ~100 along track)
        xyz_100, reg_100 = _ensure_100_points(xyz, regions, n_points=n_points)

        # Position axis for MATLAB (µm along probe)
        # Use linear spacing from min..max pos
        pos_100 = np.linspace(float(pos.min()), float(pos.max()), n_points)

        # Write outputs
        csv_path = os.path.join(out_dir, f"histology_shank{sh_i}.csv")
        npy_path = os.path.join(out_dir, f"trackcoordinates_shank{sh_i}.npy")

        df = pd.DataFrame({
            "Position": pos_100,
            "RegionAcronym": reg_100
        })
        df.to_csv(csv_path, index=False)
        np.save(npy_path, xyz_100.astype(np.float32))

        print(f"\n[OK] Shank {sh_i}")
        print(f"  xyz key:     {xyz_key}")
        print(f"  regions key: {regions_key}")
        print(f"  depths key:  {depths_key}")
        print(f"  wrote: {csv_path}")
        print(f"  wrote: {npy_path}")

        outputs.append((csv_path, npy_path))

    return outputs

if __name__ == "__main__":
    probe_pkl = r"J:\project_trainingAggression\histologyData\975826_Agg1\herbsProbeTrajectory\probes\probe 1.pkl"
    out_dir  = r"J:\project_trainingAggression\histologyData\975826_Agg1\herbsProbeTrajectory\probes"

    convert_probe_pkl(
        probe_pkl_path=probe_pkl,
        out_dir=out_dir,
        n_points=100,
        assume_um=True,
        # If auto-detection fails, set e.g.:
        # force_xyz_key="trajectory.xyz_um"
        # force_regions_key="trajectory.region_acronym"
        # force_depths_key="trajectory.depth_um"
        force_xyz_key=None,
        force_regions_key=None,
        force_depths_key=None,
    )



=== probe.pkl flattened keys (first 60) ===
type | type=str | shape=None
data.object_name | type=str | shape=None
data.probe_type_name | type=str | shape=None
data.data | type=list | shape=None
data.pieces_names | type=list | shape=None
data.ap_tilt | type=str | shape=None
data.ml_tilt | type=str | shape=None
data.insertion_coords_3d | type=ndarray | shape=(3,)
data.terminus_coords_3d | type=ndarray | shape=(3,)
data.direction | type=ndarray | shape=(3,)
data.probe_length | type=float64 | shape=()
data.dv | type=float64 | shape=()
data.ap_angle | type=float64 | shape=()
data.ml_angle | type=float64 | shape=()
data.insertion_coords | type=ndarray | shape=(3,)
data.insertion_vox | type=ndarray | shape=(3,)
data.terminus_coords | type=ndarray | shape=(3,)
data.terminus_vox | type=ndarray | shape=(3,)
data.sites_label | type=list | shape=None
data.sites_loc_b | type=list | shape=None
data.sites_vox | type=list | shape=None
data.region_label | type=list | shape=None
data.region_length | ty